<a href="https://colab.research.google.com/github/ha0-922/VLA/blob/main/0407_SmolVLA_Inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🤖 SmolVLA Inference 데모

## 모델 소개: SmolVLA
- **SmolVLA** (Small Vision-Language-Action Model)는 HuggingFace가 2025년 공개한 소형 VLA 모델
- VLA = **Vision(이미지) + Language(언어 명령) + Action(로봇 동작)** 을 하나의 모델로 처리
- 파라미터 수: **450M** (코랩 T4에서도 실행 가능)
- 내부 구조: SmolVLM2 (비전-언어 백본) + Action Expert (행동 예측 헤드)
- 학습 데이터: HuggingFace LeRobot 플랫폼의 커뮤니티 로봇 데이터셋

## 전체 흐름
```
카메라 이미지 ─┐
               ├──▶ SmolVLA ──▶ 로봇 관절 action (6차원 벡터)
언어 명령    ─┘
```

In [ ]:
!pip install lerobot num2words -q
print("설치 완료")

## Step 1. 모델 & 데이터셋 로드
- 모델: `lerobot/smolvla_base` (HuggingFace Hub)
- 데이터셋: `lerobot/libero` (SmolVLA 공식 예제 데이터셋)
- `make_pre_post_processors`: 입력 정규화, 이미지 리사이즈 등을 자동 처리

In [ ]:
import torch
from lerobot.datasets.lerobot_dataset import LeRobotDataset
from lerobot.policies.factory import make_pre_post_processors
from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("디바이스:", device)

model_id = "lerobot/smolvla_base"

# 모델 로드
policy = SmolVLAPolicy.from_pretrained(model_id).to(device).eval()

# 전처리기 로드 (정규화, 리사이즈 등 자동 처리)
preprocess, postprocess = make_pre_post_processors(
    policy.config,
    model_id,
    preprocessor_overrides={"device_processor": {"device": str(device)}},
)

# 데이터셋 로드
dataset = LeRobotDataset("lerobot/libero")

print("---- 준비 완료 ----")
print("데이터셋 크기:", len(dataset))
print("샘플 키:", list(dataset[0].keys()))

In [ ]:
# 단일 프레임 추론 테스트
frame = dict(dataset[0])
task = frame["task"]
print("언어 명령:", task)
print("원본 키:", list(frame.keys()))

# preprocess 적용
batch = preprocess(frame)
print("\n전처리 후 키:", list(batch.keys()))

In [ ]:
import torch

frame = dict(dataset[0])
task = frame["task"]
print("언어 명령:", task)

# preprocess 적용
batch = preprocess(frame)

# 이미지 키 이름 맞추기
batch["observation.images.camera1"] = batch.pop("observation.images.image").to(device)
batch["observation.images.camera2"] = batch.pop("observation.images.image2").to(device)
batch["observation.images.camera3"] = batch["observation.images.camera1"]  # 카메라 3개 중 하나 없으면 복사

# 나머지 텐서 device로 이동
batch["observation.state"] = batch["observation.state"].to(device)
batch["observation.language.tokens"] = batch["observation.language.tokens"].to(device)
batch["observation.language.attention_mask"] = batch["observation.language.attention_mask"].bool().to(device)

# 추론
with torch.no_grad():
    action = policy.select_action(batch)

action = postprocess(action)

print("✅ 추론 완료!")
print("언어 명령:", task)
print("action shape:", action.shape)
print("action:", action)

## Step 2. 연속 추론 & 시각화
- 20개 프레임에 걸쳐 연속으로 추론
- 프레임마다 다른 이미지를 입력 (실제 로봇이 카메라 보면서 판단하는 것과 동일한 구조)
- 관절 6개의 action이 시간에 따라 어떻게 변하는지 확인

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

actions = []
n_frames = 20
policy.reset()

for i in range(n_frames):
    frame = dict(dataset[i])

    batch = preprocess(frame)
    batch["observation.images.camera1"] = batch.pop("observation.images.image").to(device)
    batch["observation.images.camera2"] = batch.pop("observation.images.image2").to(device)
    batch["observation.images.camera3"] = batch["observation.images.camera1"]
    batch["observation.state"] = batch["observation.state"].to(device)
    batch["observation.language.tokens"] = batch["observation.language.tokens"].to(device)
    batch["observation.language.attention_mask"] = batch["observation.language.attention_mask"].bool().to(device)

    with torch.no_grad():
        action = policy.select_action(batch)
    action = postprocess(action)
    actions.append(action.squeeze(0).cpu().numpy())

actions = np.array(actions)  # (20, 6)

fig, axes = plt.subplots(2, 3, figsize=(12, 6))
fig.suptitle(
    "SmolVLA (lerobot/smolvla_base) Inference Result\n"
    "Dataset: lerobot/libero\n"
    f"Task: '{frame['task']}'",
    fontsize=10
)

joint_names = ["Joint 1 (X)", "Joint 2 (Y)", "Joint 3 (Z)",
               "Joint 4 (Roll)", "Joint 5 (Pitch)", "Joint 6 (Yaw)"]

for i, ax in enumerate(axes.flat):
    ax.plot(actions[:, i], marker='o', color=f"C{i}", linewidth=2)
    ax.set_title(joint_names[i])
    ax.set_xlabel("Frame (time →)")
    ax.set_ylabel("Action Value")
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("smolvla_actions.png", dpi=100)
plt.show()
print("✅ 시각화 완료!")

## Step 3. 언어 명령 비교 실험
- 동일한 이미지 시퀀스에 **다른 언어 명령**을 주면 action이 어떻게 달라지는지 비교
- 명령마다 action이 다르게 나오면 → SmolVLA가 언어를 실제로 이해한다는 증거

In [ ]:
tasks_to_compare = [
    "put the white mug on the left plate.",
    "move the arm to the left.",
    "open the gripper.",
]

all_actions = {}

for task in tasks_to_compare:
    policy.reset()
    task_actions = []

    for i in range(20):
        frame = dict(dataset[i])
        frame["task"] = task  # 언어 명령만 교체

        batch = preprocess(frame)
        batch["observation.images.camera1"] = batch.pop("observation.images.image").to(device)
        batch["observation.images.camera2"] = batch.pop("observation.images.image2").to(device)
        batch["observation.images.camera3"] = batch["observation.images.camera1"]
        batch["observation.state"] = batch["observation.state"].to(device)
        batch["observation.language.tokens"] = batch["observation.language.tokens"].to(device)
        batch["observation.language.attention_mask"] = batch["observation.language.attention_mask"].bool().to(device)

        with torch.no_grad():
            action = policy.select_action(batch)
        action = postprocess(action)
        task_actions.append(action.squeeze(0).cpu().numpy())

    all_actions[task] = np.array(task_actions)
    print(f"✅ 완료: '{task}'")

print("\nAll command inferences complete!")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
fig.suptitle(
    "SmolVLA: Action Comparison by Language Command\n"
    "(Same image sequence, different language instructions)",
    fontsize=12
)

joint_names = ["Joint 1 (X)", "Joint 2 (Y)", "Joint 3 (Z)",
               "Joint 4 (Roll)", "Joint 5 (Pitch)", "Joint 6 (Yaw)"]
colors = ["tab:blue", "tab:orange", "tab:green"]
short_labels = ["Put mug on plate", "Move left", "Open gripper"]

for j, ax in enumerate(axes.flat):
    for i, (task, actions) in enumerate(all_actions.items()):
        ax.plot(actions[:, j], color=colors[i],
                label=short_labels[i], linewidth=2, marker='o', markersize=3)
    ax.set_title(joint_names[j])
    ax.set_xlabel("Frame (time →)")
    ax.set_ylabel("Action Value")
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("smolvla_command_comparison.png", dpi=100)
plt.show()
print("✅ 비교 시각화 완료")

## Step 4. 정답 Action vs SmolVLA 예측 비교
- 데이터셋에 저장된 정답 action과 SmolVLA 예측 비교
- MAE (평균 절대 오차)로 정량 평가
- smolvla_base는 범용 모델이므로 fine-tuning 없이는 어느 정도 오차가 있는 게 자연스러운 결과

In [ ]:
gt_actions = []
pred_actions = []
policy.reset()

for i in range(20):
    frame = dict(dataset[i])

    batch = preprocess(frame)
    batch["observation.images.camera1"] = batch.pop("observation.images.image").to(device)
    batch["observation.images.camera2"] = batch.pop("observation.images.image2").to(device)
    batch["observation.images.camera3"] = batch["observation.images.camera1"]
    batch["observation.state"] = batch["observation.state"].to(device)
    batch["observation.language.tokens"] = batch["observation.language.tokens"].to(device)
    batch["observation.language.attention_mask"] = batch["observation.language.attention_mask"].bool().to(device)

    with torch.no_grad():
        action = policy.select_action(batch)
    action = postprocess(action)

    gt_actions.append(frame["action"].cpu().numpy()[:6])  # 앞 6개
    pred_actions.append(action.squeeze(0).cpu().numpy())

gt_actions = np.array(gt_actions)
pred_actions = np.array(pred_actions)

mae = np.mean(np.abs(gt_actions - pred_actions))
print(f"평균 예측 오차 (MAE): {mae:.4f}")

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
fig.suptitle(
    f"SmolVLA: Ground Truth vs Predicted Action\n"
    f"Task: '{frame['task']}' | MAE: {mae:.4f}",
    fontsize=11
)

for j, ax in enumerate(axes.flat):
    ax.plot(gt_actions[:, j], color="tab:blue",
            label="Ground Truth", linewidth=2, marker='o', markersize=3)
    ax.plot(pred_actions[:, j], color="tab:red",
            label="SmolVLA Prediction", linewidth=2,
            marker='s', markersize=3, linestyle='--')
    ax.set_title(joint_names[j])
    ax.set_xlabel("Frame (time →)")
    ax.set_ylabel("Action Value")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("smolvla_gt_vs_pred.png", dpi=100)
plt.show()
print("✅ 비교 시각화 완료")

In [ ]:
from google.colab import files
files.download("smolvla_actions.png")
files.download("smolvla_command_comparison.png")
files.download("smolvla_gt_vs_pred.png")